In [ ]:
import torch
import os
import nltk
# nltk.download('punkt_tab') # Uncomment and run this the first time the project is used on a machine.
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
import numpy as np
from gensim.models import KeyedVectors

100%|██████████| 870k/870k [00:00<00:00, 3.71MB/s]


In [ ]:
# This function creates a model using a supplied text file.
# The parameters are the name of the text file once it is downloaded, and the url is the url of the selected text.
# The output of this function is a skipgram model.
def generateEmbeddings(name, url):
  destination_path = os.path.join(os.getcwd(), name)
  torch.hub.download_url_to_file(url, destination_path)
  with open(destination_path, 'r') as file:
      corpus = file.read()

  data = []

  for i in sent_tokenize(corpus):
      temp = []

      # tokenize the sentence into words
      for j in word_tokenize(i):
          temp.append(j.lower())

      data.append(temp)

  model = gensim.models.Word2Vec(data, min_count=1, vector_size=300,
                                window=5, sg=1, seed=42, workers=1)

  return model

In [ ]:
dracula = generateEmbeddings('dracula', 'https://www.gutenberg.org/cache/epub/345/pg345.txt')
oliver = generateEmbeddings('oliver', 'https://www.gutenberg.org/cache/epub/730/pg730.txt')

100%|██████████| 870k/870k [00:00<00:00, 4.09MB/s]
100%|██████████| 933k/933k [00:00<00:00, 3.14MB/s]
100%|██████████| 1.11M/1.11M [00:00<00:00, 4.53MB/s]


In [ ]:
# This function finds the word in the model that is most similar to the supplied word.
# The parameter is the word of interest.
# The function prints out a warning if the word does not exist in the model.
def duoMostSim(word):
  try:
    dracSim = dracula.wv.most_similar(word)
    print('Dracula: ', dracSim)
  except:
    print("This word is not used in Dracula!")
  try:
    olSim = oliver.wv.most_similar(word)
    print('Oliver: ', olSim)
  except:
    print("This word does not exist in Oliver Twist!")

In [ ]:
# This function finds the similarity between the supplied words in the model.
# The parameters are the two words of interest.
# The function prints out a warning if one or both words do not exist in the model.
def duoSim(word, otherWord):
  try:
    dracSim = dracula.wv.similarity(word, otherWord)
    print("Dracula: ", dracSim)
  except:
    print("One of the supplied words is not used in Dracula!")
  try:
    olSim = oliver.wv.similarity(word, otherWord)
    print("Oliver: ", olSim)
  except:
    print("One of the supplied words is not used in Oliver Twist!")

In [ ]:
duoMostSim('child')

Dracula:  [('answer', 0.9664287567138672), ('words', 0.9662159085273743), ('sad', 0.9658651351928711), ('tired', 0.9652963280677795), ('clever', 0.9643005132675171), ('quick', 0.9641224145889282), ('simply', 0.9630707502365112), ('lady', 0.9627334475517273), ('young', 0.961290180683136), ('good-bye', 0.9605838060379028)]
Oliver:  [('mother', 0.9752336144447327), ('fellow', 0.9736077785491943), ('soul', 0.97274249792099), ('word', 0.9647374749183655), ('wretch', 0.9639519453048706), ('son', 0.9626006484031677), ('barney', 0.9623052477836609), ('undertaker', 0.9620192646980286), ('servant', 0.9613103270530701), ('named', 0.9604442119598389)]
Iliad:  [('warn', 0.9941560626029968), ('hapless', 0.9936714768409729), ('unrevenged', 0.992821991443634), ('aids', 0.9925290942192078), ('successful', 0.9923163056373596), ('calm', 0.9914957880973816), ('sparta', 0.9913172721862793), ('argives', 0.9912188649177551), ('blest', 0.9911813139915466), ('ransom', 0.9909074902534485)]


In [ ]:
duoMostSim('dog')

Dracula:  [('small', 0.99350506067276), ('chapel', 0.9833399653434753), ('half', 0.9831971526145935), ('terror', 0.9827606678009033), ('difficulty', 0.9826333522796631), ('seemingly', 0.9825344681739807), ('bat', 0.9824201464653015), ('big', 0.9823333024978638), ('sense', 0.9807006120681763), ('grim', 0.9805099964141846)]
Oliver:  [('wife', 0.9755076766014099), ('office', 0.9753661751747131), ('housebreaker', 0.9746802449226379), ('stranger', 0.9737775325775146), ('shop', 0.9729493856430054), ('coach', 0.9710601568222046), ('pause', 0.9708923101425171), ('seat', 0.9706659317016602), ('second', 0.9702646136283875), ('son', 0.9702182412147522)]
Iliad:  [('purchase', 0.9967150688171387), ('detested', 0.9966549277305603), ('ogilby', 0.9966428279876709), ('sea-god', 0.996588408946991), ('pledge', 0.9965828657150269), ('patience', 0.9965574145317078), ('deceased', 0.9960713982582092), ('spectacle', 0.9960581064224243), ('victory', 0.9959815740585327), ('unconscious', 0.9959710836410522)]


In [ ]:
duoSim('dog', 'child')

Dracula:  0.83836395
Oliver:  0.9416788
Iliad:  0.951893


In [ ]:
duoSim('water', 'house')

Dracula:  0.91033566
Oliver:  0.9155273
Iliad:  0.9402499


In [ ]:
# This function creates a clone of a model for internal use.
def clone_kv(model):
    kv = KeyedVectors(vector_size=model.vector_size)
    kv.add_vectors(model.index_to_key, model.vectors)
    return kv

# This function looks for words that are similar to the supplied word in both models.
# The output of this function is a list of the top n most similar words that exist in both models.
def getShared(word, source_model, target_vocab, n=10, search_k=50):
    #search up to k similiar words to find n words that are in both vocabs
    try:
        neighbors = source_model.most_similar(word, topn=search_k)
    except KeyError:
        return []
    valid = [w for w, _ in neighbors if w in target_vocab] #if in both
    return valid[:n] #return first n

# This function merges the two supplied models.
# The output is two new models that contain the word vectors of both of the originals.
def merge(model1: KeyedVectors, model2: KeyedVectors, n=10, search_k=50):
    #add words from each model to one another using averaged values from shared similar vectors
    vocab1 = set(model1.index_to_key)
    vocab2 = set(model2.index_to_key)

    newModel1 = clone_kv(model1)
    newModel2 = clone_kv(model2)

    #add words to 1st model
    for word in vocab2 - vocab1:
        valid = getShared(word, model2, vocab1, n=n, search_k=search_k)
        if not valid: # []
            continue
        vecs = np.array([newModel1[w] for w in valid]) #similar vectors
        newModel1.add_vector(word, vecs.mean(axis=0)) #add meaned vector

    #2nd
    for word in vocab1 - vocab2:
        valid = getShared(word, model1, vocab2, n=n, search_k=search_k)
        if not valid:
            continue
        vecs = np.array([newModel2[w] for w in valid])
        newModel2.add_vector(word, vecs.mean(axis=0))

    return newModel1, newModel2


In [ ]:
draculaOliver, oliverDracula = merge(dracula.wv, oliver.wv)

/home/dsu/.local/lib/python3.10/site-packages/gensim/models/keyedvectors.py:551: UserWarning: Adding single vectors to a KeyedVectors which grows by one each time can be costly. Consider adding in batches or preallocating to the required size.
  warnings.warn(


In [ ]:
# This block is used for testing. It checks if any words are missing from either model.
vocab1 = set(draculaOliver.index_to_key)
vocab2 = set(oliverDracula.index_to_key)

missingIn1 = vocab2 - vocab1
missingIn2 = vocab1 - vocab2

In [ ]:
# Uses the original models.
print('Dracula: ', dracula.wv.most_similar('count'))
print('Oliver: ', oliver.wv.most_similar('count'))

Dracula:  [('renfield', 0.939416766166687), ('same', 0.9216254353523254), ('castle', 0.9139893054962158), ('papers', 0.9128307700157166), ('patient', 0.9020019769668579), ('body', 0.9019068479537964), ('driver', 0.8993672132492065), ('tea', 0.8961066007614136), ('habit', 0.8954049944877625), ('dinner', 0.8943226933479309)]
Oliver:  [('visitors', 0.9965862035751343), ('avowal', 0.9964204430580139), ('scarecely', 0.9961096048355103), ('lies', 0.9960461854934692), ('stories', 0.9960162043571472), ('favourable', 0.9960123896598816), ('desert', 0.9959813952445984), ('perform', 0.9959031939506531), ('asking', 0.9958953261375427), ('gang', 0.9958901405334473)]


In [ ]:
# Uses the merged models.
print('Dracula: ', draculaOliver.most_similar('count'))
print('Oliver: ', oliverDracula.most_similar('count'))

Dracula:  [('renfield', 0.939416766166687), ('same', 0.9216254353523254), ('castle', 0.9139893054962158), ('papers', 0.9128307700157166), ('patient', 0.9020019769668579), ('body', 0.9019068479537964), ('driver', 0.8993672132492065), ('tea', 0.8961066007614136), ('habit', 0.8954049944877625), ('dinner', 0.8943226933479309)]
Oliver:  [('stinted', 0.9978228807449341), ('pacify', 0.9974704384803772), ('over-anxious', 0.9974325895309448), ('implication', 0.9973471164703369), ('condemn', 0.9973398447036743), ('contemplate', 0.9972535967826843), ('create', 0.997229278087616), ('absorb', 0.9971677660942078), ('sceptic', 0.9970836639404297), ('bulgar', 0.9970616698265076)]
